# Adapter Debiasing - Condition 3 (Parameter-Efficient)
## What this notebook does

| Property | Value |
|---|---|
| Training data | CDA-augmented (`train_augmented.csv`) |
| Trainable params | Adapter modules + classifier (~1.8M BERT / ~900K DistilBERT) |
| Backbone | Frozen |
| Fairness loss | None |
| Class weights | From `data/class_weights.json` (shared) |

## 1. Install packages

In [ ]:
# PyTorch with CUDA 12.1 for RTX 4060 Ti
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install adapters WITHOUT pinning transformers - adapters will pull its
# own compatible transformers version automatically.
# Forcing transformers==4.44.0 causes _CAN_RECORD_REGISTRY import error.
%pip install adapters --quiet

# Supporting packages
%pip install "numpy==1.26.4" datasets==2.20.0 scikit-learn pandas codecarbon scipy --quiet

## 2. Imports

In [ ]:
import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import softmax
from torch.utils.data import DataLoader

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

import adapters
from adapters import AutoAdapterModel

from codecarbon import EmissionsTracker
print("adapters version:", adapters.__version__)

## 3. Configuration

In [ ]:
MODEL_NAME = "bert-base-uncased"
# MODEL_NAME = "distilbert-base-uncased"
# MODEL_NAME = "roberta-base"

ADAPTER_NAME             = "debiasing"
ADAPTER_REDUCTION_FACTOR = 16   # bottleneck = hidden_size / 16 = 48

RUN_MODE = "STANDARD"   # "DEBUG" | "STANDARD" 

if RUN_MODE == "DEBUG":
    MAX_LENGTH    = 128
    EPOCHS        = 1
    TRAIN_BATCH   = 4
    EVAL_BATCH    = 4
elif RUN_MODE == "STANDARD":
    MAX_LENGTH    = 128
    EPOCHS        = 3
    TRAIN_BATCH   = 16
    EVAL_BATCH    = 32

# Adapter eval disabled - mid-training eval incompatible with AdapterHub output format
# Final evaluation uses manual DataLoader loop (cell 13)
SAVE_STRATEGY = "no"
LOAD_BEST     = False

BASE_DIR    = r"C:\Users\scanc\Desktop\v3"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "baseline_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

safe_model_name = MODEL_NAME.replace("/", "_").replace("-", "_") + "final_test"

print(f"Model                  : {MODEL_NAME}")
print(f"Run mode               : {RUN_MODE}")
print(f"Data dir               : {DATA_DIR}")
print(f"Results dir            : {RESULTS_DIR}")

## 4. Check device

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = "cuda"
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("No GPU")
print("Active device:", device)

## 5. Load shared splits and class weights

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_augmented.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

with open(os.path.join(DATA_DIR, "class_weights.json")) as f:
    cw = json.load(f)
class_weights_dict   = {int(k): v for k, v in cw.items()}
class_weights_tensor = torch.tensor(
    [class_weights_dict[0], class_weights_dict[1]], dtype=torch.float32
)

print(f"Train (augmented) : {len(train_df):,}")
print(f"Val               : {len(val_df):,}")
print(f"Test              : {len(test_df):,}")
print(f"Class weights     : {class_weights_dict}")

if RUN_MODE == "DEBUG":
    train_df = train_df.sample(n=min(2000, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df   = val_df.sample(n=min(200,   len(val_df)),   random_state=SEED).reset_index(drop=True)
    test_df  = test_df.sample(n=min(400,   len(test_df)),  random_state=SEED).reset_index(drop=True)
    print(f"DEBUG: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 6. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    return tokenizer(examples["comment_text"], truncation=True, max_length=MAX_LENGTH)

train_hf  = Dataset.from_pandas(train_df[["comment_text","label"]].reset_index(drop=True))
val_hf    = Dataset.from_pandas(val_df[["comment_text","label"]].reset_index(drop=True))
test_hf   = Dataset.from_pandas(test_df[["comment_text","label"]].reset_index(drop=True))

hf_ds     = DatasetDict({"train": train_hf, "validation": val_hf, "test": test_hf})
tokenized = hf_ds.map(preprocess, batched=True)

keep = ["input_ids", "attention_mask", "token_type_ids", "label"]
for split in tokenized:
    to_remove = [c for c in tokenized[split].column_names if c not in keep]
    tokenized[split] = tokenized[split].remove_columns(to_remove)
tokenized.set_format("torch")
print(tokenized)

## 7. Load model and add adapter

`AutoAdapterModel` is the AdapterHub drop-in for sequence classification.
`DoubleSeqBnConfig` inserts bottleneck adapters after both the attention
sub-layer and the FFN sub-layer in every transformer layer.
`model.train_adapter()` freezes all backbone weights automatically.

In [ ]:
model = AutoAdapterModel.from_pretrained(MODEL_NAME)

adapter_config = adapters.DoubleSeqBnConfig(reduction_factor=ADAPTER_REDUCTION_FACTOR)
model.add_adapter(ADAPTER_NAME, config=adapter_config)
model.add_classification_head(ADAPTER_NAME, num_labels=2, overwrite_ok=True)
model.train_adapter(ADAPTER_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Adapter config       : DoubleSeqBnConfig (reduction={ADAPTER_REDUCTION_FACTOR})")
print(f"Bottleneck size      : {model.config.hidden_size // ADAPTER_REDUCTION_FACTOR}")
print(f"Trainable parameters : {trainable:,}")
print(f"Total parameters     : {total:,}")
print(f"Trainable %          : {100*trainable/total:.4f}%")

## 8. Custom Trainer with weighted loss

`AdapterTrainer` uses fixed class weights from `class_weights.json`.
No fairness loss - CDA is the sole debiasing signal for Condition 3.

In [ ]:
class AdapterTrainer(Trainer):
    """Trainer with fixed class-weighted cross-entropy loss.

    No auxiliary fairness loss is used. This condition is CDA plus adapter
    tuning only.
    """

    def __init__(self, class_weights_tensor, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights_tensor.detach().clone().float()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # AdapterHub sometimes returns logits as a tuple.
        if isinstance(logits, tuple):
            logits = logits[0]
        # Handle binary / squeezed outputs.
        if logits.ndim == 1:
            logits = logits.unsqueeze(0)

        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss = nn.functional.cross_entropy(
            logits, labels.to(logits.device), weight=weights
        )
        return (loss, outputs) if return_outputs else loss

## 9. Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.array(logits, dtype=np.float32)

    if logits.ndim == 1:
        probabilities = 1 / (1 + np.exp(-logits))
        predictions   = (probabilities >= 0.5).astype(int)
    else:
        predictions   = np.argmax(logits, axis=1)
        probabilities = softmax(logits, axis=1)[:, 1]

    accuracy                = accuracy_score(labels, predictions)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, predictions, average="binary",   zero_division=0)
    _, _, f1_w, _           = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(labels, probabilities)
    except ValueError:
        auc = float("nan")

    return {
        "accuracy"    : round(accuracy, 4),
        "precision"   : round(p_bin,    4),
        "recall"      : round(r_bin,    4),
        "f1_binary"   : round(f1_bin,   4),
        "f1_weighted" : round(f1_w,     4),
        "auc_roc"     : round(auc,      4),
    }

## 10. Output folders

In [ ]:
output_dir  = os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_model")
logging_dir = os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_logs")
os.makedirs(output_dir,  exist_ok=True)
os.makedirs(logging_dir, exist_ok=True)
print("Output dir:", output_dir)

## 11. Training arguments

`eval_strategy = "no"` and `load_best_model_at_end = False` because
AdapterHub's output format causes the Trainer's metric collection to
fail during mid-training eval. Final evaluation uses a manual
DataLoader loop in cell 13.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir    = output_dir,
    eval_strategy = "no",
    save_strategy = "no",

    learning_rate               = 1e-4,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size  = EVAL_BATCH,
    num_train_epochs            = EPOCHS,
    weight_decay                = 0.01,

    logging_dir   = logging_dir,
    logging_steps = 10,

    load_best_model_at_end = False,
    metric_for_best_model  = None,

    report_to             = "none",
    seed                  = SEED,
    remove_unused_columns = False,
)

## 12. Train with CodeCarbon tracking

In [ ]:
tracker = EmissionsTracker(
    project_name = f"{safe_model_name}_adapter",
    output_dir   = RESULTS_DIR,
    output_file  = f"{safe_model_name}_adapter_emissions.csv",
    log_level    = "warning",
    save_to_file = True,
)

trainer = AdapterTrainer(
    class_weights_tensor = class_weights_tensor,
    model                = model,
    args                 = training_args,
    train_dataset        = tokenized["train"],
    eval_dataset         = tokenized["validation"],
    compute_metrics      = compute_metrics,
    data_collator        = data_collator,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

tracker.start()
wall_start = time.time()

trainer.train()

wall_end        = time.time()
emissions_kg    = tracker.stop()
runtime_seconds = wall_end - wall_start

print(f"\nTraining complete.")
print(f"  Wall-clock time : {runtime_seconds:.1f} s  ({runtime_seconds/60:.2f} min)")
print(f"  CO2 emitted     : {emissions_kg:.6f} kg CO2eq")

## 13. Manual evaluation on full test set

`trainer.predict()` with AdapterHub only processes one batch.
This manual DataLoader loop collects predictions for all test examples.

In [ ]:
print("Running manual inference on full test set...")

model.eval()
all_logits  = []
device_str  = "cuda" if torch.cuda.is_available() else "cpu"

test_loader = DataLoader(
    tokenized["test"],
    batch_size = EVAL_BATCH,
    collate_fn = data_collator,
    shuffle    = False,
)

with torch.no_grad():
    for batch in test_loader:
        # Remove labels before passing to model
        batch    = {k: v.to(device_str) for k, v in batch.items() if k != "labels"}
        outputs  = model(**batch)
        logits_b = outputs.logits
        if isinstance(logits_b, tuple):
            logits_b = logits_b[0]
        all_logits.append(logits_b.cpu().float().numpy())

logits_all = np.concatenate(all_logits, axis=0)
labels_all = np.array(tokenized["test"]["label"])

print(f"Logits shape : {logits_all.shape}")
print(f"Labels shape : {labels_all.shape}")
assert len(logits_all) == len(labels_all), "Mismatch between predictions and labels!"

if logits_all.ndim == 1:
    probs_all = 1 / (1 + np.exp(-logits_all))
else:
    probs_all = softmax(logits_all, axis=1)[:, 1]

preds_all = (probs_all >= 0.5).astype(int)

p, r, f1, _  = precision_recall_fscore_support(labels_all, preds_all, average="binary",   zero_division=0)
_, _, f1w, _ = precision_recall_fscore_support(labels_all, preds_all, average="weighted", zero_division=0)

results = {
    "eval_accuracy"    : round(accuracy_score(labels_all, preds_all), 4),
    "eval_precision"   : round(p,   4),
    "eval_recall"      : round(r,   4),
    "eval_f1_binary"   : round(f1,  4),
    "eval_f1_weighted" : round(f1w, 4),
    "eval_auc_roc"     : round(roc_auc_score(labels_all, probs_all), 4),
}

print("\nTest set results:")
for k, v in results.items():
    print(f"  {k:<30} {v}")

## 14. Save performance results

In [ ]:
results_df = pd.DataFrame([results])
results_df["model"]             = MODEL_NAME
results_df["method"]            = "adapter"
results_df["run_mode"]          = RUN_MODE
results_df["train_size"]        = len(train_df)
results_df["val_size"]          = len(val_df)
results_df["test_size"]         = len(test_df)
results_df["cda"]               = True
results_df["fairness_loss"]     = False
results_df["adapter_reduction"] = ADAPTER_REDUCTION_FACTOR
results_df["max_length"]        = MAX_LENGTH
results_df["epochs"]            = EPOCHS
results_df["runtime_seconds"]   = runtime_seconds
results_df["runtime_minutes"]   = runtime_seconds / 60

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_results.csv")
results_df.to_csv(path, index=False)
print("Saved:", path)
results_df

## 15. Save predictions

In [ ]:
# Reuse logits_all, probs_all, preds_all, labels_all from cell 13
prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_label"]     = labels_all
prediction_df["toxicity_score"] = probs_all
prediction_df["prediction"]     = preds_all
prediction_df["model"]          = MODEL_NAME
prediction_df["method"]         = "adapter"

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_predictions.csv")
prediction_df.to_csv(path, index=False)
print("Saved:", path)
print(f"Rows : {len(prediction_df):,}")
prediction_df.head()

## 16. Save sustainability metrics

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
peak_gpu  = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else None

try:
    cc_df      = pd.read_csv(os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_emissions.csv"))
    energy_kwh = cc_df["energy_consumed"].iloc[-1]
except Exception:
    energy_kwh = None

sustainability_df = pd.DataFrame([{
    "model"                 : MODEL_NAME,
    "method"                : "adapter",
    "cda"                   : True,
    "fairness_loss"         : False,
    "adapter_reduction"     : ADAPTER_REDUCTION_FACTOR,
    "adapter_bottleneck"    : model.config.hidden_size // ADAPTER_REDUCTION_FACTOR,
    "run_mode"              : RUN_MODE,
    "train_size"            : len(train_df),
    "val_size"              : len(val_df),
    "test_size"             : len(test_df),
    "runtime_seconds"       : round(runtime_seconds, 2),
    "runtime_minutes"       : round(runtime_seconds / 60, 4),
    "energy_kwh"            : energy_kwh,
    "emissions_kg_co2eq"    : round(emissions_kg, 8) if emissions_kg else None,
    "trainable_parameters"  : trainable,
    "total_parameters"      : total,
    "trainable_pct"         : round(100 * trainable / total, 4),
    "peak_gpu_memory_gb"    : round(peak_gpu, 4) if peak_gpu else None,
    "max_length"            : MAX_LENGTH,
    "epochs"                : EPOCHS,
}])

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_adapter_sustainability.csv")
sustainability_df.to_csv(path, index=False)
print("Saved:", path)
sustainability_df